# Manchester United: Finding the Perfect Casemiro Replacement
**Data source:** Understat (xG-focused) + curated scouting stats  
**Role profile:** Defensive midfielder (CDM / no. 6)  
**Candidates:** Carlos Baleba, Aurélien Tchouaméni, Éderson, Sandro Tonali, Adam Wharton

## 1. Setup & Dependencies

In [ ]:
# Install required packages (run once)
# !pip install understatapi pandas matplotlib seaborn numpy nest_asyncio

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

print('Libraries loaded OK')

## 2. Fetch Live Data from Understat

This section pulls EPL player data for the 2024/25 season and filters to our shortlist.  
**Understat player IDs** (used for direct player lookups):

| Player | Understat ID |
|---|---|
| Casemiro | 7592 |
| Carlos Baleba | 11756 |
| Sandro Tonali | 8378 |
| Adam Wharton | 12801 |

In [ ]:
from understatapi import UnderstatClient
import pandas as pd

UNDERSTAT_IDS = {
    'Casemiro': '7592',
    'Baleba':   '11756',
    'Tonali':   '8378',
    'Wharton':  '12801',
}

def fetch_all_players():
    client = UnderstatClient()
    results = {}
    for name, pid in UNDERSTAT_IDS.items():
        try:
            shots = client.player(player=pid).get_shot_data()
            results[name] = pd.DataFrame(shots)
            print(f'  {name}: {len(shots)} shot events fetched')
        except Exception as e:
            print(f'  {name}: fetch failed — {e}')
    return results

print('Fetching Understat shot data...')
shot_data = fetch_all_players()
print('Done.')

In [ ]:
# Preview Casemiro's shot data
if 'Casemiro' in shot_data:
    df_case = shot_data['Casemiro']
    print(df_case.columns.tolist())
    df_case[['minute', 'result', 'xG', 'situation', 'season']].head(10)

## 3. Scouting Dataset (curated stats)

Understat is xG-focused (attacking/shot data). For a complete CDM comparison we combine it with defensive stats from public sources (FBref/Sofascore). This cell holds the curated scouting data that powers the analysis.

In [ ]:
# Curated scouting data — 2024/25 season stats
# Sources: Sofascore, FBref, Fotmob, transfer reports (May 2026)

scouting_data = [
    {
        'player':          'Casemiro',
        'club':            'Man United',
        'age':             34,
        'nationality':     'Brazil',
        'fee_estimate_m':  0,            # outgoing, no fee
        'minutes':         2507,
        'goals':           9,
        'assists':         2,
        'avg_rating':      7.36,
        'tackles_p90':     2.1,
        'interceptions_p90': 1.6,
        'tackle_success_pct': 71,
        'pass_acc_pct':    86,
        'duels_won_pct':   52,
        'xG':              3.2,
        'xA':              1.1,
        'yellow_cards':    8,
        'red_cards':       1,
    },
    {
        'player':          'Baleba',
        'club':            'Brighton',
        'age':             22,
        'nationality':     'Cameroon',
        'fee_estimate_m':  78,
        'minutes':         2310,
        'goals':           2,
        'assists':         3,
        'avg_rating':      7.21,
        'tackles_p90':     2.8,
        'interceptions_p90': 2.0,
        'tackle_success_pct': 74,
        'pass_acc_pct':    86,
        'duels_won_pct':   57,
        'xG':              0.9,
        'xA':              1.4,
        'yellow_cards':    6,
        'red_cards':       0,
    },
    {
        'player':          'Tchouaméni',
        'club':            'Real Madrid',
        'age':             26,
        'nationality':     'France',
        'fee_estimate_m':  87,
        'minutes':         2650,
        'goals':           3,
        'assists':         2,
        'avg_rating':      7.15,
        'tackles_p90':     2.3,
        'interceptions_p90': 1.8,
        'tackle_success_pct': 69,
        'pass_acc_pct':    88,
        'duels_won_pct':   55,
        'xG':              1.4,
        'xA':              0.9,
        'yellow_cards':    5,
        'red_cards':       0,
    },
    {
        'player':          'Éderson',
        'club':            'Atalanta',
        'age':             26,
        'nationality':     'Brazil',
        'fee_estimate_m':  55,
        'minutes':         2700,
        'goals':           2,
        'assists':         4,
        'avg_rating':      7.28,
        'tackles_p90':     2.4,
        'interceptions_p90': 1.9,
        'tackle_success_pct': 74,
        'pass_acc_pct':    89,
        'duels_won_pct':   54,
        'xG':              1.1,
        'xA':              1.8,
        'yellow_cards':    7,
        'red_cards':       0,
    },
    {
        'player':          'Tonali',
        'club':            'Newcastle',
        'age':             24,
        'nationality':     'Italy',
        'fee_estimate_m':  80,
        'minutes':         2100,
        'goals':           3,
        'assists':         5,
        'avg_rating':      7.10,
        'tackles_p90':     1.9,
        'interceptions_p90': 1.6,
        'tackle_success_pct': 65,
        'pass_acc_pct':    87,
        'duels_won_pct':   51,
        'xG':              1.3,
        'xA':              2.1,
        'yellow_cards':    8,
        'red_cards':       0,
    },
    {
        'player':          'Wharton',
        'club':            'Crystal Palace',
        'age':             21,
        'nationality':     'England',
        'fee_estimate_m':  62,
        'minutes':         2340,
        'goals':           1,
        'assists':         3,
        'avg_rating':      7.05,
        'tackles_p90':     2.5,
        'interceptions_p90': 1.7,
        'tackle_success_pct': 64,
        'pass_acc_pct':    91,
        'duels_won_pct':   48,
        'xG':              0.4,
        'xA':              1.3,
        'yellow_cards':    4,
        'red_cards':       0,
    },
]

df = pd.DataFrame(scouting_data)

# Derived columns
df['defensive_actions_p90'] = df['tackles_p90'] + df['interceptions_p90']
df['goal_contributions']    = df['goals'] + df['assists']
df['xG_plus_xA']            = df['xG'] + df['xA']
df['years_at_prime']        = (30 - df['age']).clip(lower=0)  # years left in prime (under 30)

print(df[['player', 'club', 'age', 'tackles_p90', 'pass_acc_pct', 'fee_estimate_m']].to_string(index=False))

## 4. Defensive Profile Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Defensive profile — all candidates vs Casemiro baseline', fontsize=14, fontweight='bold', y=1.01)

colors = ['#e63946' if p == 'Casemiro' else '#457b9d' for p in df['player']]

metrics = [
    ('tackles_p90',            'Tackles per 90'),
    ('interceptions_p90',      'Interceptions per 90'),
    ('tackle_success_pct',     'Tackle success %'),
]

for ax, (col, label) in zip(axes, metrics):
    bars = ax.barh(df['player'], df[col], color=colors, edgecolor='white', linewidth=0.5)
    ax.set_xlabel(label)
    ax.set_title(label)
    for bar, val in zip(bars, df[col]):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', va='center', fontsize=9)

red_patch  = mpatches.Patch(color='#e63946', label='Casemiro (baseline)')
blue_patch = mpatches.Patch(color='#457b9d', label='Candidates')
fig.legend(handles=[red_patch, blue_patch], loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.savefig('defensive_profile.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Radar Chart — Overall CDM Profile

In [ ]:
from matplotlib.patches import Polygon
import matplotlib.cm as cm

# Normalize all metrics to 0-1 for radar
radar_metrics = [
    ('tackles_p90',          'Tackling'),
    ('interceptions_p90',    'Interceptions'),
    ('tackle_success_pct',   'Tackle success'),
    ('pass_acc_pct',         'Passing'),
    ('duels_won_pct',        'Duels won'),
    ('avg_rating',           'Rating'),
]

cols    = [m[0] for m in radar_metrics]
labels  = [m[1] for m in radar_metrics]
N       = len(labels)

df_norm = df.copy()
for col in cols:
    mn, mx = df[col].min(), df[col].max()
    df_norm[col] = (df[col] - mn) / (mx - mn) if mx > mn else 0.5

angles  = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

palette = ['#e63946', '#2196f3', '#ff9800', '#4caf50', '#9c27b0', '#00bcd4']

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_facecolor('#f8f8f8')

for i, row in df_norm.iterrows():
    vals = [row[c] for c in cols]
    vals += vals[:1]
    color = palette[i % len(palette)]
    ax.plot(angles, vals, color=color, linewidth=2, label=row['player'])
    ax.fill(angles, vals, color=color, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8, color='gray')
ax.set_title('CDM profile radar — normalized stats', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)

plt.tight_layout()
plt.savefig('radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. xG Analysis from Understat

Uses the live shot data fetched in Section 2. For CDMs, we care less about volume shooting and more about **shot quality when they do shoot** (high xG per shot = composed finisher) and **xA** (progressive, chance-creating passing).

In [ ]:
# Build xG summary from Understat shot data (if fetched successfully)
xg_rows = []

for name, shots_df in shot_data.items():
    if shots_df.empty:
        continue
    shots_df['xG'] = pd.to_numeric(shots_df.get('xG', 0), errors='coerce').fillna(0)
    n_shots    = len(shots_df)
    total_xg   = shots_df['xG'].sum()
    xg_per_shot = total_xg / n_shots if n_shots > 0 else 0
    xg_rows.append({'player': name, 'n_shots': n_shots, 'total_xG': round(total_xg, 2),
                    'xG_per_shot': round(xg_per_shot, 3)})

if xg_rows:
    df_xg = pd.DataFrame(xg_rows)
    print(df_xg.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Understat xG breakdown (EPL-available players)', fontsize=13, fontweight='bold')

    axes[0].bar(df_xg['player'], df_xg['total_xG'], color='#457b9d')
    axes[0].set_title('Total xG')
    axes[0].set_ylabel('xG')

    axes[1].bar(df_xg['player'], df_xg['xG_per_shot'], color='#2a9d8f')
    axes[1].set_title('xG per shot (shot quality)')
    axes[1].set_ylabel('xG / shot')

    plt.tight_layout()
    plt.savefig('xg_breakdown.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No Understat data fetched — run Section 2 first, or check your connection.')

## 7. Value-for-Money Analysis

In [ ]:
candidates = df[df['player'] != 'Casemiro'].copy()

# Composite defensive score (weighted sum)
candidates['defensive_score'] = (
    candidates['tackles_p90']          * 0.30 +
    candidates['interceptions_p90']    * 0.25 +
    candidates['tackle_success_pct']   * 0.003 +   # scale to ~0-3 range
    candidates['duels_won_pct']        * 0.003 +
    candidates['pass_acc_pct']         * 0.002 +
    candidates['avg_rating']           * 0.10
)

fig, ax = plt.subplots(figsize=(9, 6))

scatter_colors = ['#2196f3', '#ff9800', '#4caf50', '#9c27b0', '#00bcd4']
for i, (_, row) in enumerate(candidates.iterrows()):
    ax.scatter(row['fee_estimate_m'], row['defensive_score'],
               s=200, color=scatter_colors[i], zorder=5, edgecolors='white', linewidth=1.5)
    ax.annotate(row['player'],
                xy=(row['fee_estimate_m'], row['defensive_score']),
                xytext=(6, 4), textcoords='offset points', fontsize=10)

# Casemiro reference lines
case = df[df['player'] == 'Casemiro'].iloc[0]
case_score = (
    case['tackles_p90'] * 0.30 +
    case['interceptions_p90'] * 0.25 +
    case['tackle_success_pct'] * 0.003 +
    case['duels_won_pct'] * 0.003 +
    case['pass_acc_pct'] * 0.002 +
    case['avg_rating'] * 0.10
)
ax.axhline(case_score, color='#e63946', linestyle='--', linewidth=1.2,
           label=f'Casemiro baseline ({case_score:.2f})')

ax.set_xlabel('Transfer fee estimate (€m)', fontsize=12)
ax.set_ylabel('Composite defensive score', fontsize=12)
ax.set_title('Value for money: defensive output vs transfer cost', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('value_for_money.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Composite Scouting Score & Final Ranking

In [ ]:
# Normalize each metric to 0-100 scale, then apply weights
def normalize(series):
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn) * 100 if mx > mn else pd.Series([50] * len(series), index=series.index)

weights = {
    'tackles_p90':          0.20,
    'interceptions_p90':    0.15,
    'tackle_success_pct':   0.15,
    'pass_acc_pct':         0.15,
    'duels_won_pct':        0.10,
    'avg_rating':           0.10,
    'years_at_prime':       0.10,  # age/future value
    'xG_plus_xA':          0.05,
}

df_score = df.copy()
df_score['composite_score'] = sum(
    normalize(df_score[col]) * w for col, w in weights.items()
)

# Fee efficiency bonus: higher score / lower fee = better value
df_score_candidates = df_score[df_score['player'] != 'Casemiro'].copy()
df_score_candidates = df_score_candidates.sort_values('composite_score', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))

bar_colors = ['#2196f3' if i == 0 else '#90caf9' for i in range(len(df_score_candidates))]
bars = ax.barh(df_score_candidates['player'], df_score_candidates['composite_score'],
               color=bar_colors, edgecolor='white')

for bar, (_, row) in zip(bars, df_score_candidates.iterrows()):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{row['composite_score']:.1f}  |  €{int(row['fee_estimate_m'])}m  |  age {int(row['age'])}",
            va='center', fontsize=9, color='#333')

ax.set_xlabel('Composite scouting score (0–100)', fontsize=11)
ax.set_title('Final ranking — Casemiro replacement shortlist', fontsize=13, fontweight='bold')
ax.set_xlim(0, 115)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('final_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Final recommendation ---')
winner = df_score_candidates.iloc[0]
print(f"Top pick: {winner['player']} ({winner['club']}, age {int(winner['age'])})")
print(f"Score:    {winner['composite_score']:.1f} / 100")
print(f"Fee est:  €{int(winner['fee_estimate_m'])}m")

## 9. Correlation Heatmap

In [ ]:
corr_cols = ['tackles_p90', 'interceptions_p90', 'tackle_success_pct',
             'pass_acc_pct', 'duels_won_pct', 'avg_rating', 'xG_plus_xA', 'fee_estimate_m']

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle only
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Stat correlation matrix — CDM candidate pool', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary

Run the cell below to print a clean summary report.

In [ ]:
print('=' * 60)
print('MANCHESTER UNITED — CASEMIRO REPLACEMENT ANALYSIS')
print('=' * 60)

ranked = df_score_candidates[['player', 'club', 'age', 'composite_score', 'fee_estimate_m']].copy()
ranked.columns = ['Player', 'Club', 'Age', 'Score', 'Fee (€m)']
ranked['Score'] = ranked['Score'].round(1)
ranked = ranked.reset_index(drop=True)
ranked.index += 1
print(ranked.to_string())

print('\nKey insights:')
top3 = ranked.head(3)
for rank, row in top3.iterrows():
    print(f"  {rank}. {row['Player']} — score {row['Score']}, age {int(row['Age'])}, ~€{int(row['Fee (€m)'])}m")

print('\nData sources: Understat (xG), Sofascore/FBref (defensive stats), transfer reports (May 2026)')
print('=' * 60)